# MedGemma word-level analysis

Tokens -> words (medical-NER labelled), then AOPC, the per-word drop
distribution, and the Friedman test.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import wordlevel as wl

CSV = "data/per_token_drops_medgemma.csv"
TOKENIZER = "google/medgemma-1.5-4b-it"
METHODS = ["attention", "gmar_l1", "gmar_l2", "gradcam", "random"]

In [ ]:
tokens = pd.read_csv(CSV)
tokens.shape, sorted(tokens["method"].unique()), sorted(tokens["mask_ratio"].unique())

### Reconstruct words

In [ ]:
tok = AutoTokenizer.from_pretrained(TOKENIZER)
words = wl.reconstruct_words(tokens, tok)
print(len(words), "words")
words.head()

### NER labels

In [ ]:
words = wl.annotate(words)
words.to_csv("words_medgemma.csv", index=False)
words["ner_class"].value_counts()

### AOPC by method and mask ratio

In [ ]:
wl.aopc_table(words)

### Per-word drop distribution at r = 0.1

In [ ]:
wl.distribution_stats(words, mask_ratio=0.1)

### Friedman test per (mask ratio, NER class), BH-corrected

In [ ]:
fr = wl.friedman_by_cell(words, METHODS)
print((fr["q_value"] < 0.05).sum(), "/", len(fr), "strata significant at q<0.05")
fr

### AOPC by NER class

In [ ]:
wl.ner_class_table(words)